# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. The dataset represents mass spectrometry analyses of extracellular vesicles isolated from stimulated human mast cells, with rich metadata and data organized according to the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an mlcroissant.DatasetMetadata object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), fields, and their IDs (or `@id`).

For each record set, we list its `@id` and show its fields (columns/attributes with their `@id`s and descriptions if available).

In [ ]:
# Find all record sets (`cr:RecordSet`)
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    # List columns/fields
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) type: {field.data_type} desc: {field.description}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below we:
- Choose the main protein record set by its `@id` (see the output above).
- Load its records to a pandas DataFrame.
- Preview its columns and first records.

*Replace `<record_set_id>` below with the actual `@id` string for the record set you wish to analyze. All further fields are referenced by their `@id`s.*

In [ ]:
# List available record set IDs to select from
record_set_ids = [rs.id for rs in dataset.record_sets()]
print("RecordSet @ids:", record_set_ids)

# Select the main data table (likely the first/main one in the dataset)
main_record_set_id = record_set_ids[0]  # or select by name/description
print(f"Selected record set: {main_record_set_id}")

# Extract records for each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

print("Columns in selected DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data by key attributes.

We demonstrate:
- Filtering proteins with high molecular weight (MW)
- Normalizing a numerical column
- Grouping by a categorical variable (such as 'Description' if available)

*All columns referenced by their `@id` as listed previously.*

In [ ]:
# --- Inspect fields to choose appropriate numeric and group columns ---
df = dataframes[main_record_set_id]
print("All columns:", df.columns.tolist())

# Try to find some numeric columns and a categorical column to use by looking at field names
numeric_candidates = [c for c in df.columns if (
    df[c].dtype in (float, int) or 'weight' in c.lower() or 'abundance' in c.lower() or 'coverage' in c.lower())]
print(f"Numeric candidates: {numeric_candidates}")
# Select a likely numeric column
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.select_dtypes(include='number').columns[0]

# Try to find a group/categoric field
group_candidates = [c for c in df.columns if (df[c].dtype == object and 'desc' in c.lower() or 'type' in c.lower())]
if group_candidates:
    group_field = group_candidates[0]
else:
    group_field = df.columns[0]

# --- Filtering and Normalization ---
try:
    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with '{numeric_field}' > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
except Exception as e:
    print(f"Error during EDA: {e}")

## 5. Visualization
Visualize distributions of selected fields, such as molecular weight or normalized abundance, and relationships (such as group-wise means).

*Requires matplotlib.*

In [ ]:
import matplotlib.pyplot as plt

if numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

if group_field in df.columns and numeric_field in df.columns:
    plt.figure(figsize=(10, 5))
    df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).plot(kind='bar')
    plt.ylabel(f"Mean of {numeric_field}")
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load and explore the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We loaded record sets by their `@id`, inspected fields, filtered and normalized key numeric fields, grouped by categoricals, and visualized data distributions.

For your own use case, reference all columns and record sets by their `@id`; see the earlier outputs in this notebook for those identifiers. For further analysis, replace `main_record_set_id`, `numeric_field`, and `group_field` as appropriate for the fields you wish to analyze.